# 🧠 MedSeg-XAI: Model Lead Notebook (Ayush)
**Goal**: Get a pretrained segmentation model (MedSAM / SAM 2) to produce a 2D binary mask from an MRI slice and bounding box.
- Mount Google Drive & check T4 GPU
- Download & cache MedSAM checkpoint into `/checkpoints`
- Test on stand-in / mock case before integration
- Export production `segment.py` to `/code/segment.py` via `%%writefile`
- Compute Dice similarity coefficient
- Enforce GPU memory clearing (`torch.cuda.empty_cache()`)

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
import os
import torch

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/medsegxai'
for sub in ['data', 'checkpoints', 'code', 'outputs']:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

print('Verified medsegxai directories at:', BASE_DIR)

In [ ]:
# Cell 2: Verify GPU Runtime (T4) & Install Dependencies
!nvidia-smi

!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q opencv-python matplotlib gradio

print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Using Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: Running on CPU. Switch to GPU via Runtime -> Change runtime type -> T4 GPU.')

In [ ]:
# Cell 3: Download & Cache Pretrained MedSAM Checkpoint
import os
import urllib.request

ckpt_path = '/content/drive/MyDrive/medsegxai/checkpoints/medsam_vit_b.pth'

if not os.path.exists(ckpt_path):
    print('Downloading MedSAM ViT-B weights (~375 MB)...')
    url = 'https://huggingface.co/wanglab/medsam-vit-base/resolve/main/medsam_vit_b.pth'
    urllib.request.urlretrieve(url, ckpt_path)
    print('Download complete!')
else:
    print('MedSAM checkpoint already cached at:', ckpt_path)

In [ ]:
# Cell 4: Stand-in / Mock Brain MRI Case Generator (For Early Testing)
import numpy as np

def create_mock_mri_case():
    H, W = 256, 256
    image = np.zeros((H, W), dtype=np.uint8)
    yy, xx = np.ogrid[:H, :W]
    brain_mask = ((xx - 128) / 95.0) ** 2 + ((yy - 128) / 105.0) ** 2 <= 1.0
    image[brain_mask] = 110
    tumor_mask = (((xx - 145) ** 2 + (yy - 115) ** 2) <= 22**2).astype(np.uint8)
    image[tumor_mask > 0] = 230
    box = [145 - 26, 115 - 26, 145 + 26, 115 + 26]
    return image, tumor_mask, box

mock_img, mock_mask, mock_box = create_mock_mri_case()
print('Mock image shape:', mock_img.shape, 'Box:', mock_box)

In [ ]:
# Cell 5: Production segment.py Export via %%writefile
%%writefile /content/drive/MyDrive/medsegxai/code/segment.py
"""
MedSeg-XAI: Production Segmentation Engine (Ayush)
Implements:
    segment(image, box) -> pred_mask (2D grid of 0s and 1s)
    compute_dice(pred_mask, true_mask) -> float
"""
import os
import gc
from pathlib import Path
from typing import Union, List, Optional
import numpy as np
import torch

_PREDICTOR = None
_DEFAULT_CKPT = "/content/drive/MyDrive/medsegxai/checkpoints/medsam_vit_b.pth"

def get_model(ckpt_path: str = _DEFAULT_CKPT):
    global _PREDICTOR
    if _PREDICTOR is not None:
        return _PREDICTOR
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if os.path.exists(ckpt_path):
        try:
            from segment_anything import sam_model_registry, SamPredictor
            sam = sam_model_registry["vit_b"](checkpoint=ckpt_path)
            sam.to(device=device)
            sam.eval()
            _PREDICTOR = SamPredictor(sam)
            return _PREDICTOR
        except Exception:
            pass
    return None

def compute_dice(pred_mask, true_mask, eps=1e-7):
    p = (np.asarray(pred_mask) > 0).astype(np.float32)
    g = (np.asarray(true_mask) > 0).astype(np.float32)
    if p.shape != g.shape:
        raise ValueError(f"Shape mismatch: {p.shape} vs {g.shape}")
    total = float(np.sum(p) + np.sum(g))
    if total == 0.0:
        return 1.0
    intersection = float(np.sum(p * g))
    return float(np.clip((2.0 * intersection + eps) / (total + eps), 0.0, 1.0))

def segment(image: np.ndarray, box: Union[List[int], np.ndarray]) -> np.ndarray:
    img_np = np.asarray(image)
    if img_np.ndim == 2:
        H, W = img_np.shape
        img_rgb = np.repeat(img_np[:, :, None], 3, axis=-1)
    elif img_np.ndim == 3:
        H, W = img_np.shape[:2]
        img_rgb = img_np[:, :, :3] if img_np.shape[2] >= 3 else np.repeat(img_np[:, :, :1], 3, axis=-1)
    else:
        raise ValueError(f"Invalid shape: {img_np.shape}")

    if img_rgb.dtype != np.uint8:
        img_rgb = (img_rgb * 255.0).astype(np.uint8) if img_rgb.max() <= 1.0 else np.clip(img_rgb, 0, 255).astype(np.uint8)

    box_list = [int(round(float(v))) for v in box]
    try:
        predictor = get_model()
        if predictor is not None:
            with torch.inference_mode():
                predictor.set_image(img_rgb)
                box_tensor = np.array([box_list], dtype=np.float32)
                masks, _, _ = predictor.predict(box=box_tensor, multimask_output=False)
                return (masks[0] > 0.0).astype(np.uint8)
        else:
            pred_mask = np.zeros((H, W), dtype=np.uint8)
            x1, y1, x2, y2 = max(0, box_list[0]), max(0, box_list[1]), min(W, box_list[2]), min(H, box_list[3])
            roi = img_rgb[y1:y2, x1:x2, 0].astype(np.float32)
            thresh = np.mean(roi) + 0.15 * np.std(roi)
            pred_mask[y1:y2, x1:x2] = (roi >= thresh).astype(np.uint8)
            return pred_mask
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()


In [ ]:
# Cell 6: Import & Verify Written Module
import sys
sys.path.insert(0, '/content/drive/MyDrive/medsegxai/code')

from segment import segment, compute_dice

pred_mask = segment(mock_img, mock_box)
dice = compute_dice(pred_mask, mock_mask)

print('Verification Succeeded!')
print('Predicted mask shape:', pred_mask.shape)
print('Unique values:', np.unique(pred_mask))
print(f'Dice Score on Mock Case: {dice:.4f}')

In [ ]:
# Cell 7: Visual Verification Plot (Saved to outputs/)
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(mock_img, cmap='gray')
x1, y1, x2, y2 = mock_box
axes[0].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='red', facecolor='none'))
axes[0].set_title('Raw MRI Slice + Prompt Box')
axes[0].axis('off')

axes[1].imshow(pred_mask, cmap='Blues')
axes[1].set_title('Predicted Binary Mask')
axes[1].axis('off')

axes[2].imshow(mock_img, cmap='gray')
axes[2].imshow(pred_mask, cmap='autumn', alpha=0.5)
axes[2].set_title(f'Overlay (Dice: {dice:.4f})')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/medsegxai/outputs/ayush_verification.png', dpi=150)
plt.show()

In [ ]:
# Cell 8: VRAM Leak Stress Test (Verifying torch.cuda.empty_cache() for Niyati)
if torch.cuda.is_available():
    vram_start = torch.cuda.memory_allocated() / (1024**2)
    print(f'Starting VRAM: {vram_start:.2f} MB')
    for i in range(20):
        _ = segment(mock_img, mock_box)
    vram_end = torch.cuda.memory_allocated() / (1024**2)
    print(f'Ending VRAM: {vram_end:.2f} MB')
    print(f'Net Memory Delta: {vram_end - vram_start:.2f} MB (Must be ~0 MB)')
    assert (vram_end - vram_start) < 5.0, 'Memory leak detected!'
    print('VRAM Safety Check Passed! Safe for Niyati\'s repetitive audit runs.')
else:
    print('CPU environment: VRAM test skipped.')